In [1]:
#!/usr/bin/env python3
"""
==============================================================================
Bayesian Network Inference — Aviation Causal Risk Model
==============================================================================

Question Types:
  Q1. Forward Propagation   — "What happens downstream if X occurs?"
  Q2. Backward Inference    — "What likely caused outcome Y?"
  Q3. Sensitivity Analysis  — "Which factor has the greatest impact?"
  Q4. What-If / Intervention— "How much does fixing X reduce risk of Y?"
==============================================================================
"""

import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict
from functools import reduce

# ============================================================================
# 1. MODEL DEFINITION
# ============================================================================

topic_labels = {
    'topic_0':  'Passenger & Crew Injuries',
    'topic_1':  'Aircraft Structural Damage',
    'topic_2':  'ATC Communication Issues',
    'topic_4':  'Landing Gear Malfunctions',
    'topic_6':  'Fatigue Crack',
    'topic_7':  'Turbulence Encounter',
    'topic_8':  'Fuel System Installation Issues',
    'topic_9':  'Runway & Taxiway Incidents',
    'topic_10': 'Electrical Wiring Damage',
    'topic_11': 'Fan Blade Failure',
    'topic_13': 'Aircraft Electrical System Faults',
    'topic_14': 'High EGT',
    'topic_15': 'Turbine & Compressor Damage',
    'topic_16': 'Smoke or Odor in Cabin',
    'topic_18': 'Hydraulic System Pressure Issues',
}

dag_edges = [
    ('topic_9', 'topic_0'), ('topic_9', 'topic_1'), ('topic_7', 'topic_1'),
    ('topic_2', 'topic_7'), ('topic_4', 'topic_9'), ('topic_13', 'topic_4'),
    ('topic_6', 'topic_13'), ('topic_8', 'topic_13'), ('topic_8', 'topic_6'),
    ('topic_8', 'topic_10'), ('topic_8', 'topic_11'), ('topic_10', 'topic_16'),
    ('topic_10', 'topic_18'), ('topic_13', 'topic_16'), ('topic_14', 'topic_15'),
    ('topic_14', 'topic_11'), ('topic_14', 'topic_10'), ('topic_11', 'topic_15'),
    ('topic_6', 'topic_11'),
]

parents_of = defaultdict(list)
children_of = defaultdict(list)
all_nodes = set()
for p, c in dag_edges:
    parents_of[c].append(p)
    children_of[p].append(c)
    all_nodes.add(p); all_nodes.add(c)

root_nodes = sorted([n for n in all_nodes if not parents_of[n]])
topo_order = ['topic_14','topic_2','topic_8','topic_7','topic_6',
              'topic_10','topic_13','topic_11','topic_18','topic_4',
              'topic_16','topic_15','topic_9','topic_0','topic_1']

# BDeu CPTs: cpts[node][(parent_vals)] = P(node=1 | parents)
cpts = {
    'topic_2':  {(): 0.2215},
    'topic_8':  {(): 0.0638},
    'topic_14': {(): 0.0554},
    'topic_0':  {(0,): 0.5764, (1,): 0.3870},
    'topic_7':  {(0,): 0.1742, (1,): 0.0465},
    'topic_9':  {(0,): 0.0784, (1,): 0.1971},
    'topic_4':  {(0,): 0.1364, (1,): 0.3364},
    'topic_6':  {(0,): 0.0558, (1,): 0.2531},
    'topic_18': {(0,): 0.0357, (1,): 0.2463},
    'topic_1':  {(0,0): 0.2355, (0,1): 0.6033, (1,0): 0.1268, (1,1): 0.1190},
    'topic_10': {(0,0): 0.0293, (0,1): 0.2064, (1,0): 0.2674, (1,1): 0.4394},
    'topic_13': {(0,0): 0.0276, (0,1): 0.1860, (1,0): 0.1800, (1,1): 0.1585},
    'topic_15': {(0,0): 0.0228, (0,1): 0.1327, (1,0): 0.2419, (1,1): 0.6224},
    'topic_16': {(0,0): 0.0397, (0,1): 0.1989, (1,0): 0.1923, (1,1): 0.3824},
    'topic_11': {
        (0,0,0): 0.0085, (0,0,1): 0.1199, (0,1,0): 0.2249, (0,1,1): 0.9138,
        (1,0,0): 0.0979, (1,0,1): 0.2681, (1,1,0): 0.3491, (1,1,1): 0.8077,
    },
}


def lbl(t):
    return topic_labels.get(t, t)


# ============================================================================
# 2. TRY PGMPY — FALL BACK TO PURE-PYTHON EXACT INFERENCE
# ============================================================================

USE_PGMPY = False

try:
    from pgmpy.models import BayesianNetwork
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    model = BayesianNetwork(dag_edges)

    # Build TabularCPDs
    for node in topo_order:
        pars = parents_of[node]
        if not pars:
            p1 = cpts[node][()]
            cpd = TabularCPD(node, 2, [[1 - p1], [p1]])
        else:
            n_combos = 2 ** len(pars)
            col0 = []  # P(node=0|...)
            col1 = []  # P(node=1|...)
            for combo in product([0, 1], repeat=len(pars)):
                p1 = cpts[node][combo]
                col0.append(1 - p1)
                col1.append(p1)
            cpd = TabularCPD(
                node, 2, [col0, col1],
                evidence=pars,
                evidence_card=[2] * len(pars)
            )
        model.add_cpds(cpd)

    assert model.check_model()
    infer = VariableElimination(model)
    USE_PGMPY = True
    print("Using pgmpy VariableElimination (exact inference)\n")

except Exception as e:
    print(f"pgmpy not available ({e})")
    print("Using pure-Python exact Variable Elimination (factor-based)\n")


# ============================================================================
# 3. PURE-PYTHON EXACT INFERENCE (Variable Elimination with factors)
# ============================================================================

class Factor:
    """A discrete factor over binary variables."""
    def __init__(self, variables, values):
        """
        variables: list of variable names
        values: dict mapping tuple of (0/1) assignments -> probability
        """
        self.variables = list(variables)
        self.values = dict(values)

    def restrict(self, evidence):
        """Return a new factor with evidence variables fixed."""
        ev_vars = set(evidence.keys()) & set(self.variables)
        if not ev_vars:
            return Factor(self.variables, self.values)
        new_vars = [v for v in self.variables if v not in ev_vars]
        new_values = {}
        for assignment, prob in self.values.items():
            asgn_dict = dict(zip(self.variables, assignment))
            consistent = all(asgn_dict[v] == evidence[v] for v in ev_vars)
            if consistent:
                new_key = tuple(asgn_dict[v] for v in new_vars)
                new_values[new_key] = prob
        return Factor(new_vars, new_values)

    def multiply(self, other):
        """Multiply two factors."""
        all_vars = list(dict.fromkeys(self.variables + other.variables))
        new_values = {}
        for combo in product([0, 1], repeat=len(all_vars)):
            asgn = dict(zip(all_vars, combo))
            key_self = tuple(asgn[v] for v in self.variables)
            key_other = tuple(asgn[v] for v in other.variables)
            if key_self in self.values and key_other in other.values:
                new_values[combo] = self.values[key_self] * other.values[key_other]
        return Factor(all_vars, new_values)

    def marginalize(self, var):
        """Sum out a variable."""
        if var not in self.variables:
            return Factor(self.variables, self.values)
        var_idx = self.variables.index(var)
        new_vars = [v for v in self.variables if v != var]
        new_values = {}
        for assignment, prob in self.values.items():
            new_key = tuple(v for i, v in enumerate(assignment) if i != var_idx)
            new_values[new_key] = new_values.get(new_key, 0.0) + prob
        return Factor(new_vars, new_values)

    def normalize(self):
        """Normalize so values sum to 1."""
        total = sum(self.values.values())
        if total > 0:
            return Factor(self.variables, {k: v / total for k, v in self.values.items()})
        return self


def build_factors():
    """Build a factor for each node from CPTs."""
    factors = []
    for node in topo_order:
        pars = parents_of[node]
        variables = pars + [node]
        values = {}
        if not pars:
            p1 = cpts[node][()]
            values[(0,)] = 1 - p1
            values[(1,)] = p1
            factors.append(Factor([node], values))
        else:
            for combo in product([0, 1], repeat=len(pars)):
                p1 = cpts[node][combo]
                values[combo + (1,)] = p1
                values[combo + (0,)] = 1 - p1
            factors.append(Factor(variables, values))
    return factors


def query_exact(query_var, evidence=None):
    """
    Exact inference via Variable Elimination.
    Returns P(query_var=1 | evidence).
    """
    if evidence is None:
        evidence = {}
    
    factors = build_factors()
    
    # Apply evidence
    factors = [f.restrict(evidence) for f in factors]
    
    # Determine elimination order (all vars except query)
    all_vars_in_factors = set()
    for f in factors:
        all_vars_in_factors.update(f.variables)
    
    elim_vars = [v for v in topo_order if v in all_vars_in_factors
                 and v != query_var and v not in evidence]
    
    # Eliminate variables one by one
    for var in elim_vars:
        # Collect factors involving var
        involved = [f for f in factors if var in f.variables]
        remaining = [f for f in factors if var not in f.variables]
        
        if involved:
            # Multiply all involved factors
            product_factor = involved[0]
            for f in involved[1:]:
                product_factor = product_factor.multiply(f)
            # Marginalize out var
            new_factor = product_factor.marginalize(var)
            remaining.append(new_factor)
        
        factors = remaining
    
    # Multiply remaining factors
    if factors:
        result = factors[0]
        for f in factors[1:]:
            result = result.multiply(f)
        result = result.normalize()
        
        # Extract P(query_var=1)
        for assignment, prob in result.values.items():
            asgn_dict = dict(zip(result.variables, assignment))
            if asgn_dict.get(query_var, None) == 1:
                return prob
    return 0.0


def query_all(evidence=None):
    """Query P(node=1|evidence) for all nodes."""
    if evidence is None:
        evidence = {}
    result = {}
    for node in topo_order:
        if node in evidence:
            result[node] = float(evidence[node])
        else:
            result[node] = query_exact(node, evidence)
    return result


def infer_query(query_var, evidence):
    """Unified interface: use pgmpy if available, else pure-Python."""
    if USE_PGMPY:
        result = infer.query([query_var], evidence=evidence)
        return result.values[1]
    else:
        return query_exact(query_var, evidence)


def infer_all(evidence):
    """Query all nodes given evidence."""
    if USE_PGMPY:
        result = {}
        for node in topo_order:
            if node in evidence:
                result[node] = float(evidence[node])
            else:
                r = infer.query([node], evidence=evidence)
                result[node] = r.values[1]
        return result
    else:
        return query_all(evidence)


# ============================================================================
# 4. COMPUTE PRIORS
# ============================================================================
print("=" * 85)
print("PRIOR PROBABILITIES (no evidence)")
print("=" * 85)

prior = infer_all({})
print(f"\n  {'Node':<12} {'Label':<40} {'P(X=1)':>8}")
print(f"  {'-'*12} {'-'*40} {'-'*8}")
for node in topo_order:
    print(f"  {node:<12} {lbl(node):<40} {prior[node]:>8.4f}")


# ============================================================================
# HELPER: Print comparison table
# ============================================================================
def print_comparison(posteriors, prior, evidence, title=""):
    if title:
        print(f"\n  {title}")
    print(f"  {'Node':<12} {'Label':<35} {'Prior':>7} {'Post.':>7} {'Δ':>8} {'Note'}")
    print(f"  {'-'*12} {'-'*35} {'-'*7} {'-'*7} {'-'*8} {'-'*15}")
    
    # Sort by absolute change for readability
    rows = []
    for node in topo_order:
        pr = prior[node]
        po = posteriors[node]
        delta = po - pr
        if node in evidence:
            note = "◆ EVIDENCE"
        elif abs(delta) > 0.05:
            note = "▲ HIGH" if delta > 0 else "▼ HIGH"
        elif abs(delta) > 0.01:
            note = "↑" if delta > 0 else "↓"
        else:
            note = ""
        rows.append((node, pr, po, delta, note))
    
    for node, pr, po, delta, note in rows:
        print(f"  {node:<12} {lbl(node):<35} {pr:>7.4f} {po:>7.4f} {delta:>+8.4f} {note}")



c:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\inference\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pgmpy not available (BayesianNetwork has been deprecated. Please use DiscreteBayesianNetwork instead.)
Using pure-Python exact Variable Elimination (factor-based)

PRIOR PROBABILITIES (no evidence)

  Node         Label                                      P(X=1)
  ------------ ---------------------------------------- --------
  topic_14     High EGT                                   0.0554
  topic_2      ATC Communication Issues                   0.2215
  topic_8      Fuel System Installation Issues            0.0638
  topic_7      Turbulence Encounter                       0.1459
  topic_6      Fatigue Crack                              0.0684
  topic_10     Electrical Wiring Damage                   0.0543
  topic_13     Aircraft Electrical System Faults          0.0452
  topic_11     Fan Blade Failure                          0.0368
  topic_18     Hydraulic System Pressure Issues           0.0471
  topic_4      Landing Gear Malfunctions                  0.1454
  topic_16     Smoke 

In [ ]:

# ============================================================================
# Q1. FORWARD PROPAGATION
# ============================================================================
print("\n\n" + "=" * 85)
print("Q1. FORWARD PROPAGATION")
print('    "If Fuel System Installation Issues occur, what downstream risks increase?"')
print("=" * 85)

ev_q1 = {'topic_8': 1}
post_q1 = infer_all(ev_q1)
print_comparison(post_q1, prior, ev_q1,
    "Evidence: Fuel System Installation Issues = OBSERVED")


# ============================================================================
# Q1b. FORWARD PROPAGATION — High EGT
# ============================================================================
print("=" * 85)
print("Q1b. FORWARD PROPAGATION")
print('     "If High EGT is observed, what downstream risks increase?"')
print("=" * 85)

ev_q1b = {'topic_14': 1}
post_q1b = infer_all(ev_q1b)
print_comparison(post_q1b, prior, ev_q1b,
    "Evidence: High EGT = OBSERVED")





Q1. FORWARD PROPAGATION
    "If Fuel System Installation Issues occur, what downstream risks increase?"

  Evidence: Fuel System Installation Issues = OBSERVED
  Node         Label                                 Prior   Post.        Δ Note
  ------------ ----------------------------------- ------- ------- -------- ---------------
  topic_14     High EGT                             0.0554  0.0554  +0.0000 
  topic_2      ATC Communication Issues             0.2215  0.2215  +0.0000 
  topic_8      Fuel System Installation Issues      0.0638  1.0000  +0.9362 ◆ EVIDENCE
  topic_7      Turbulence Encounter                 0.1459  0.1459  -0.0000 
  topic_6      Fatigue Crack                        0.0684  0.2531  +0.1847 ▲ HIGH
  topic_10     Electrical Wiring Damage             0.0543  0.2769  +0.2226 ▲ HIGH
  topic_13     Aircraft Electrical System Faults    0.0452  0.1790  +0.1338 ▲ HIGH
  topic_11     Fan Blade Failure                    0.0368  0.1589  +0.1221 ▲ HIGH
  topic_18     

In [ ]:
# ============================================================================
# Q2. BACKWARD INFERENCE
# ============================================================================
print("\n\n" + "=" * 85)
print("Q2. BACKWARD INFERENCE (Diagnostic Reasoning)")
print('    "Smoke detected in cabin — what were the most likely root causes?"')
print("=" * 85)

ev_q2 = {'topic_16': 1}
post_q2 = infer_all(ev_q2)
print_comparison(post_q2, prior, ev_q2,
    "Evidence: Smoke or Odor in Cabin = OBSERVED")


# ============================================================================
# Q2b. BACKWARD — Turbine Damage
# ============================================================================
print("=" * 85)
print("Q2b. BACKWARD INFERENCE")
print('     "Turbine & Compressor Damage observed — what caused it?"')
print("=" * 85)

ev_q2b = {'topic_15': 1}
post_q2b = infer_all(ev_q2b)
print_comparison(post_q2b, prior, ev_q2b,
    "Evidence: Turbine & Compressor Damage = OBSERVED")





Q2. BACKWARD INFERENCE (Diagnostic Reasoning)
    "Smoke detected in cabin — what were the most likely root causes?"

  Evidence: Smoke or Odor in Cabin = OBSERVED
  Node         Label                                 Prior   Post.        Δ Note
  ------------ ----------------------------------- ------- ------- -------- ---------------
  topic_14     High EGT                             0.0554  0.0811  +0.0257 ↑
  topic_2      ATC Communication Issues             0.2215  0.2215  -0.0000 
  topic_8      Fuel System Installation Issues      0.0638  0.1292  +0.0654 ▲ HIGH
  topic_7      Turbulence Encounter                 0.1459  0.1459  -0.0000 
  topic_6      Fatigue Crack                        0.0684  0.1021  +0.0337 ↑
  topic_10     Electrical Wiring Damage             0.0543  0.2041  +0.1498 ▲ HIGH
  topic_13     Aircraft Electrical System Faults    0.0452  0.1775  +0.1322 ▲ HIGH
  topic_11     Fan Blade Failure                    0.0368  0.0548  +0.0180 ↑
  topic_18     Hydraulic

In [ ]:

# ============================================================================
# Q3. SENSITIVITY ANALYSIS
# ============================================================================
print("\n\n" + "=" * 85)
print("Q3. SENSITIVITY ANALYSIS")
print('    "Which root cause has the greatest impact on each leaf outcome?"')
print("=" * 85)

leaf_nodes = ['topic_0', 'topic_1', 'topic_15', 'topic_16', 'topic_18']

# Collect all results in a table
sens_results = []

for leaf in leaf_nodes:
    print(f"\n  ── Target: {leaf} ({lbl(leaf)}) | Prior = {prior[leaf]:.4f} ──")
    print(f"  {'Root Cause':<12} {'Label':<40} {'P(Y|X=1)':>9} {'P(Y|X=0)':>9} {'Δ(X=1)':>8} {'Δ(X=0)':>8}")
    print(f"  {'-'*12} {'-'*40} {'-'*9} {'-'*9} {'-'*8} {'-'*8}")
    
    for root in root_nodes:
        # P(leaf | root=1)
        p_given_1 = infer_query(leaf, {root: 1})
        # P(leaf | root=0)
        p_given_0 = infer_query(leaf, {root: 0})
        delta_1 = p_given_1 - prior[leaf]
        delta_0 = p_given_0 - prior[leaf]
        
        bar = "█" * max(1, int(abs(delta_1) * 200))
        print(f"  {root:<12} {lbl(root):<40} {p_given_1:>9.4f} {p_given_0:>9.4f} {delta_1:>+8.4f} {delta_0:>+8.4f}  {bar}")
        
        sens_results.append({
            'target': leaf, 'target_label': lbl(leaf),
            'root': root, 'root_label': lbl(root),
            'prior': prior[leaf],
            'P(Y|X=1)': p_given_1, 'P(Y|X=0)': p_given_0,
            'delta_X1': delta_1, 'delta_X0': delta_0,
        })





Q3. SENSITIVITY ANALYSIS
    "Which root cause has the greatest impact on each leaf outcome?"

  ── Target: topic_0 (Passenger & Crew Injuries) | Prior = 0.5583 ──
  Root Cause   Label                                     P(Y|X=1)  P(Y|X=0)   Δ(X=1)   Δ(X=0)
  ------------ ---------------------------------------- --------- --------- -------- --------
  topic_14     High EGT                                    0.5583    0.5583  -0.0000  -0.0000  █
  topic_2      ATC Communication Issues                    0.5583    0.5583  -0.0000  -0.0000  █
  topic_8      Fuel System Installation Issues             0.5577    0.5583  -0.0006  +0.0000  █

  ── Target: topic_1 (Aircraft Structural Damage) | Prior = 0.2735 ──
  Root Cause   Label                                     P(Y|X=1)  P(Y|X=0)   Δ(X=1)   Δ(X=0)
  ------------ ---------------------------------------- --------- --------- -------- --------
  topic_14     High EGT                                    0.2735    0.2735  +0.0000  +0.0000  █

In [ ]:

# ============================================================================
# Q4. WHAT-IF / INTERVENTION ANALYSIS
# ============================================================================
print("\n" + "=" * 85)
print("Q4. WHAT-IF / INTERVENTION ANALYSIS")
print('    "If we eliminate a root cause, how much does each outcome risk decrease?"')
print("=" * 85)

targets = {
    'topic_15': 'Turbine & Compressor Damage',
    'topic_16': 'Smoke or Odor in Cabin',
    'topic_18': 'Hydraulic System Pressure Issues',
    'topic_0':  'Passenger & Crew Injuries',
    'topic_1':  'Aircraft Structural Damage',
}

interventions = [
    ("No intervention (baseline)", {}),
    ("Eliminate High EGT", {'topic_14': 0}),
    ("Eliminate Fuel System Issues", {'topic_8': 0}),
    ("Improve ATC Communication", {'topic_2': 0}),
    ("Eliminate High EGT + Fuel System", {'topic_14': 0, 'topic_8': 0}),
    ("Eliminate ALL 3 root causes", {'topic_14': 0, 'topic_8': 0, 'topic_2': 0}),
]

# Compute all intervention effects
int_results = {}
for name, ev in interventions:
    int_results[name] = {}
    for target in targets:
        p = infer_query(target, ev)
        int_results[name][target] = p

# Print results per target
for target, target_lbl in targets.items():
    baseline = int_results["No intervention (baseline)"][target]
    print(f"\n  ── {target} ({target_lbl}) ──")
    print(f"  {'Intervention':<45} {'P(Y)':>7} {'ΔP':>8} {'% Reduction':>12}")
    print(f"  {'-'*45} {'-'*7} {'-'*8} {'-'*12}")
    
    for name, ev in interventions:
        p = int_results[name][target]
        if not ev:
            print(f"  {name:<45} {p:>7.4f} {'---':>8} {'---':>12}")
        else:
            dp = p - baseline
            pct = (-dp / baseline * 100) if baseline > 0 else 0
            print(f"  {name:<45} {p:>7.4f} {dp:>+8.4f} {pct:>+11.1f}%")




Q5. WHAT-IF / INTERVENTION ANALYSIS
    "If we eliminate a root cause, how much does each outcome risk decrease?"

  ── topic_15 (Turbine & Compressor Damage) ──
  Intervention                                     P(Y)       ΔP  % Reduction
  --------------------------------------------- ------- -------- ------------
  No intervention (baseline)                     0.0431      ---          ---
  Eliminate High EGT                             0.0253  -0.0178       +41.3%
  Eliminate Fuel System Issues                   0.0420  -0.0011        +2.6%
  Improve ATC Communication                      0.0431  +0.0000        -0.0%
  Eliminate High EGT + Fuel System               0.0244  -0.0187       +43.4%
  Eliminate ALL 3 root causes                    0.0244  -0.0187       +43.4%

  ── topic_16 (Smoke or Odor in Cabin) ──
  Intervention                                     P(Y)       ΔP  % Reduction
  --------------------------------------------- ------- -------- ------------
  No intervent

In [6]:


# ============================================================================
# EXPORT RESULTS
# ============================================================================
import os
output_dir = r'C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Causa_Inference'
os.makedirs(output_dir, exist_ok=True)

# Sensitivity table
sens_df = pd.DataFrame(sens_results)
sens_df.to_csv(os.path.join(output_dir, 'sensitivity_analysis.csv'), index=False)

# Intervention table
int_rows = []
for name, ev in interventions:
    row = {'intervention': name, 'evidence': str(ev)}
    for target in targets:
        row[f'P({target})'] = int_results[name][target]
    int_rows.append(row)
pd.DataFrame(int_rows).to_csv(os.path.join(output_dir, 'intervention_analysis.csv'), index=False)

# Prior table
prior_df = pd.DataFrame([{'node': n, 'label': lbl(n), 'P(X=1)': prior[n]} for n in topo_order])
prior_df.to_csv(os.path.join(output_dir, 'prior_probabilities.csv'), index=False)

print(f"\nExported to {output_dir}:")
print(f"  • sensitivity_analysis.csv")
print(f"  • intervention_analysis.csv")
print(f"  • prior_probabilities.csv")

print("\n" + "=" * 85)
print("DONE — All queries answered with exact inference.")
print("=" * 85)
print("""
  To use with pgmpy on your own machine:
    pip install pgmpy
    python bn_inference_pgmpy.py
  
  The script will automatically use pgmpy's VariableElimination
  (exact, deterministic) when available.
""")


Exported to C:\Users\55279\Desktop\Mestrado\0.Thesis\Codes\Causa_Inference:
  • sensitivity_analysis.csv
  • intervention_analysis.csv
  • prior_probabilities.csv

DONE — All queries answered with exact inference.

  To use with pgmpy on your own machine:
    pip install pgmpy
    python bn_inference_pgmpy.py

  The script will automatically use pgmpy's VariableElimination
  (exact, deterministic) when available.

